# Colab Session: repro_s1_lab08
Generated from colab-cli history log.

In [ ]:
# @title [Environment & Reproducibility Metadata]
import sys, platform, torch, transformers
print("=== Reproducibility Environment Info ===")
print("Python version      :", sys.version.split()[0])
print("Platform            :", platform.platform())
print("PyTorch version     :", torch.__version__)
print("Transformers version:", transformers.__version__)
print("CUDA available      :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device          :", torch.cuda.get_device_name(0))
    print("CUDA version        :", torch.version.cuda)
print("Default Random Seed : 42 (random.seed(42), torch.manual_seed(42))")
print("Stage 1 Model Dtype : torch.bfloat16")


=== Reproducibility Environment Info ===
Python version      : 3.13.16
Platform            : Linux-6.6.159+-x86_64-with-glibc2.39
PyTorch version     : 2.11.0+cu130
Transformers version: 5.20.2
CUDA available      : True
GPU Device          : NVIDIA L4
CUDA version        : 13.0
Default Random Seed : 42 (random.seed(42), torch.manual_seed(42))
Stage 1 Model Dtype : torch.bfloat16


In [ ]:
# ==============================================================================
# ORIGINAL EXECUTED SCRIPT EXTRACTED VERBATIM FROM TRANSCRIPT (AST-UNESCAPED)
# Transcript Step Index : 397
# Timestamp             : 2026-10-30T06:35:49Z
# Original Colab Session: stage1_iq_density_bench
# ==============================================================================

import os, sys, time, math, random, subprocess
import torch
import torch.nn as nn
import torch.nn.functional as F

if not os.path.exists('/content/minimind'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/jingyaogong/minimind.git', '/content/minimind'], check=True)
sys.path.insert(0, '/content/minimind')
os.makedirs('/content/minimind/out', exist_ok=True)

from transformers import AutoTokenizer
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM

tokenizer = AutoTokenizer.from_pretrained('/content/minimind/model')
MASK_ID = 3
SLOT_PAD_ID = 4
BOS_ID = tokenizer.bos_token_id
EOS_ID = tokenizer.eos_token_id
STYLE_DETAILED_ID = 5
STYLE_CONCISE_ID = 6

# ==============================================================================
# GRAND BENCHMARK DATASET:
# - 6 Domains x 4 States = 24 Causal Knowledge Clauses (2Seen Domains, 4 Unseen Domains!)
# - 6 Exact Rote Memories (Phone numbers, IP address, Server Key)
# Total Testable Knowledge Units = 6 Rote + 2 Seen Chains + 4 Unseen Analogy Chains = 12 Tasks (or 20 clauses)
# ==============================================================================
rote_facts = [
    ("请问妈妈的手机号码是？", "0912-888-168"),
    ("请问妈妈的家里电话是？", "02-2345-6789"),
    ("请问爸爸的手机号码是？", "0933-777-999"),
    ("请问爸爸的家里电话是？", "04-8765-4321"),
    ("请问核心服务器的IP地址是？", "192.168.10.254"),
    ("请问系统管理员的通行密钥是？", "KEY-9901-X7B")
]

# 2 Seen Domains (Model sees full causal sequence s1->s2->s3->s4)
# 4 Unseen Domains (Model NEVER sees causal transitions s2->s3->s4! Only isolated dictionary clauses!)
all_domains = [
    ("水利工程", True, [
        "水流在水管中稳定流动，", "当水管内壁结垢变窄时，", "管内水压就会急剧升高，", "最终导致水流堵塞中断。"
    ]),
    ("交通工程", True, [
        "车流在高速路正常行驶，", "当施工导致车道变窄时，", "路段交通压力急剧升高，", "最终导致车流瘫痪堵塞。"
    ]),
    ("人體醫學", False, [
        "血液在血管中正常循环，", "当血管壁堆积胆固醇变窄时，", "血管内血压就会急剧升高，", "最终导致血液循环受阻。"
    ]),
    ("網路通訊", False, [
        "数据在光纤中高速传输，", "当网络带宽通道受限变窄时，", "网络延迟与拥塞急剧升高，", "最终导致数据包传输中断。"
    ]),
    ("工廠產線", False, [
        "零件在流水线顺畅运转，", "当关键工位故障产能变窄时，", "产线积压負荷急剧升高，", "最终导致工厂生产停摆。"
    ]),
    ("呼吸系統", False, [
        "空气在气管中顺畅进出，", "当支气管发炎肿胀变窄时，", "呼吸道阻力就会急剧升高，", "最终导致呼吸困难窒息。"
    ])
]

SLOT_LEN = 12
PROMPT_LEN = 18

def pad_slot(txt, length=SLOT_LEN):
    ids = tokenizer(txt, add_special_tokens=False).input_ids[:length]
    val = [1]*len(ids) + [0]*(length - len(ids))
    pad = ids + [SLOT_PAD_ID]*(length - len(ids))
    return torch.tensor([pad], device='cuda'), torch.tensor([val], device='cuda')

def pad_prompt(txt, length=PROMPT_LEN):
    ids = [BOS_ID] + tokenizer(txt, add_special_tokens=False).input_ids[:length - 1]
    ids = [SLOT_PAD_ID]*(length - len(ids)) + ids
    return torch.tensor([ids], device='cuda')

def clean_str(t_ids):
    c = [tid for tid in t_ids.tolist() if tid not in (MASK_ID, SLOT_PAD_ID, BOS_ID, EOS_ID)]
    return tokenizer.decode(c)

# Prepare Tensors
rote_data = [(pad_prompt(q), pad_slot(a)[0], q, a) for q, a in rote_facts]
# Also add the 2 seen domain premises into Fast Muscle Memory
for d_name, is_seen, clauses in all_domains:
    if is_seen:
        rote_data.append((pad_prompt(clauses[0] + clauses[1]), pad_slot(clauses[2])[0], clauses[0] + clauses[1], clauses[2]))

dom_data = []
for d_name, is_seen, clauses in all_domains:
    c_tensors = [pad_slot(c) for c in clauses]
    dom_data.append((d_name, is_seen, clauses, c_tensors))

# ==============================================================================
# MODEL DEFINITIONS FOR THE 5-WAY BENCHMARK:
# 1. Vanilla AR-MiniMind (30.0M params, 8L, 512D)
# 2. Pure Diffusion-MiniMind (30.0M params, 8L, 512D - LLaDA style, no Thought layer)
# 3. Pure Thought-Only MiniMind (30.5M params, 8L, 512D - LCM style, forces Rote Facts through fuzzy 64-D thought bottleneck!)
# 4. Our Dual-Brain MiniMind [30.5M params, 8L, 512D]
# 5. Our TINY Dual-Brain MiniMind [5.6M params, 4L, 256D - 5.4x SMALLER!]
# ==============================================================================

class PureDiffusionMiniMind(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.backbone = MiniMindForCausalLM(config)
        for layer in self.backbone.model.layers:
            layer.self_attn.is_causal = False

    def forward_loss(self, prompt_ids, target_slots):
        B, L = target_slots.shape
        m = (torch.rand((B, L), device=target_slots.device) < 0.6)
        m[:, 0] = True
        canv = target_slots.clone(); canv[m] = MASK_ID
        lbl = target_slots.clone(); lbl[~m] = -100
        full_in = torch.cat([prompt_ids, canv], dim=1)
        logits = self.backbone(full_in).logits[:, prompt_ids.size(1):, :].contiguous()
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), lbl.reshape(-1), ignore_index=-100)

    @torch.no_grad()
    def generate_slots(self, prompt_ids, steps=2):
        B = prompt_ids.size(0)
        canv = torch.full((B, SLOT_LEN), MASK_ID, dtype=torch.long, device=prompt_ids.device)
        for s in range(steps):
            logits = self.backbone(torch.cat([prompt_ids, canv], dim=1)).logits[:, prompt_ids.size(1):, :]
            conf, pred = F.softmax(logits, dim=-1).max(dim=-1)
            still_m = (canv == MASK_ID)
            n_rem = still_m[0].sum().item()
            if n_rem == 0: break
            conf = conf.masked_fill(~still_m, -1.0)
            k = n_rem if s == steps - 1 else max(1, SLOT_LEN // steps)
            _, top_idx = torch.topk(conf, k=min(k, n_rem), dim=-1)
            for b in range(B):
                for idx in top_idx[b]:
                    if still_m[b, idx]: canv[b, idx] = pred[b, idx]
        return canv

class DualBrainMiniMind(nn.Module):
    def __init__(self, config: MiniMindConfig, slot_len: int = SLOT_LEN, pure_fuzzy_bottleneck: bool = False):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.half_dim = config.hidden_size // 2
        self.slot_len = slot_len
        self.pure_fuzzy_bottleneck = pure_fuzzy_bottleneck

        self.backbone = MiniMindForCausalLM(config)
        for layer in self.backbone.model.layers:
            layer.self_attn.is_causal = False

        # If pure_fuzzy_bottleneck=True (simulating pure LCM without Muscle Memory), compress into a tight 16-D bottleneck!
        thought_dim = 16 if pure_fuzzy_bottleneck else self.half_dim
        self.thought_dim = thought_dim
        self.domain_proj = nn.Linear(config.hidden_size, thought_dim)
        self.struct_proj = nn.Linear(config.hidden_size, thought_dim)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=thought_dim, nhead=4, dim_feedforward=thought_dim*2,
            batch_first=True, norm_first=True
        )
        self.struct_brain = nn.TransformerEncoder(enc_layer, num_layers=2, enable_nested_tensor=False)
        self.next_struct_head = nn.Linear(thought_dim, thought_dim)

        self.thought_to_prefix = nn.Linear(thought_dim * 2, config.hidden_size * 2)
        self._active_prefix = None

        def _hook(module, inp, out):
            if self._active_prefix is not None:
                out = out.clone()
                out[:, :2, :] = self._active_prefix
            return out

        self.backbone.model.embed_tokens.register_forward_hook(_hook)

    def forward_fast_muscle(self, prompt_ids, canvas_ids):
        self._active_prefix = None
        P = prompt_ids.size(1)
        full_ids = torch.cat([prompt_ids, canvas_ids], dim=1)
        return self.backbone(full_ids).logits[:, P:, :].contiguous()

    def encode_thought(self, clause_ids, valid_mask):
        self._active_prefix = None
        h = self.backbone.model(clause_ids)[0]
        mask_f = valid_mask.unsqueeze(-1).type_as(h)
        pooled = (h * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1.0)
        d = F.normalize(self.domain_proj(pooled), p=2, dim=-1)
        s = F.normalize(self.struct_proj(pooled), p=2, dim=-1)
        return d, s

    def forward_struct_hop(self, s_seq):
        B, T, _ = s_seq.shape
        c_mask = torch.triu(torch.ones(T, T, device=s_seq.device, dtype=torch.bool), diagonal=1)
        h = self.struct_brain(s_seq, mask=c_mask)
        return F.normalize(self.next_struct_head(h), p=2, dim=-1)

    def forward_thought_render(self, d, s, canvas_ids):
        B, L = canvas_ids.shape
        z = torch.cat([d, s], dim=-1)
        self._active_prefix = self.thought_to_prefix(z).reshape(B, 2, self.hidden_size)
        dummy_prefix = torch.full((B, 2), BOS_ID, dtype=torch.long, device=canvas_ids.device)
        full_ids = torch.cat([dummy_prefix, canvas_ids], dim=1)
        logits = self.backbone(full_ids).logits[:, 2:, :].contiguous()
        self._active_prefix = None
        return logits

    @torch.no_grad()
    def probe_muscle(self, prompt_ids, steps=2):
        B = prompt_ids.size(0)
        L = self.slot_len
        canv = torch.full((B, L), MASK_ID, dtype=torch.long, device=prompt_ids.device)
        min_conf = 0.0
        for s in range(steps):
            logits = self.forward_fast_muscle(prompt_ids, canv)
            probs = F.softmax(logits, dim=-1)
            conf, pred = probs.max(dim=-1)
            if s == 0:
                non_pad = (pred != SLOT_PAD_ID)
                min_conf = conf[non_pad].min().item() if non_pad.any() else conf.min().item()
            still_m = (canv == MASK_ID)
            n_rem = still_m[0].sum().item()
            if n_rem == 0: break
            conf_m = conf.masked_fill(~still_m, -1.0)
            k = n_rem if s == steps - 1 else max(1, L // steps)
            _, top_idx = torch.topk(conf_m, k=min(k, n_rem), dim=-1)
            for b in range(B):
                for idx in top_idx[b]:
                    if still_m[b, idx]: canv[b, idx] = pred[b, idx]
        return min_conf, canv

    @torch.no_grad()
    def solve_thought_analogy(self, c1_ids, c1_val, c2_ids, c2_val, steps=2):
        d1, s1 = self.encode_thought(c1_ids, c1_val)
        d2, s2 = self.encode_thought(c2_ids, c2_val)
        d_tgt = F.normalize(d1 + d2, p=2, dim=-1)
        s_seq = torch.cat([s1, s2], dim=0).unsqueeze(0)
        s3 = self.forward_struct_hop(s_seq)[:, -1, :]
        L = self.slot_len
        canv = torch.full((1, L), MASK_ID, dtype=torch.long, device=c1_ids.device)
        for step in range(steps):
            logits = self.forward_thought_render(d_tgt, s3, canv)
            conf, pred = F.softmax(logits, dim=-1).max(dim=-1)
            still_m = (canv == MASK_ID)
            n_rem = still_m[0].sum().item()
            if n_rem == 0: break
            conf_m = conf.masked_fill(~still_m, -1.0)
            k = n_rem if step == steps - 1 else max(1, L // steps)
            _, top_idx = torch.topk(conf_m, k=min(k, n_rem), dim=-1)
            for idx in top_idx[0]:
                if still_m[0, idx]: canv[0, idx] = pred[0, idx]
        return canv

# Instantiate all 5 models
cfg_512 = MiniMindConfig(hidden_size=512, num_hidden_layers=8, use_moe=False)
cfg_256 = MiniMindConfig(hidden_size=256, num_hidden_layers=4, use_moe=False)

torch.manual_seed(42)
m_ar = MiniMindForCausalLM(cfg_512).cuda().bfloat16()
torch.manual_seed(42)
m_diff = PureDiffusionMiniMind(cfg_512).cuda().bfloat16()
torch.manual_seed(42)
m_pure_thought = DualBrainMiniMind(cfg_512, pure_fuzzy_bottleneck=True).cuda().bfloat16()
torch.manual_seed(42)
m_dual_512 = DualBrainMiniMind(cfg_512, pure_fuzzy_bottleneck=False).cuda().bfloat16()
torch.manual_seed(42)
m_dual_256 = DualBrainMiniMind(cfg_256, pure_fuzzy_bottleneck=False).cuda().bfloat16()

def count_m(m): return sum(p.numel() for p in m.parameters()) / 1e6
print(f"Model Sizes: AR-512={count_m(m_ar):.2f}M | Diff-512={count_m(m_diff):.2f}M | PureThought-512={count_m(m_pure_thought):.2f}M | DualBrain-512={count_m(m_dual_512):.2f}M | Tiny-DualBrain-256={count_m(m_dual_256):.2f}M", flush=True)

# Build AR & Pure-Diffusion training sequences (Rote facts + 2 Seen Chains + Isolated clauses of 4 Unseen Domains)
ar_seqs = []
diff_pairs = []
for pt, at, q_s, a_s in rote_data:
    ids = [BOS_ID] + tokenizer(q_s + a_s, add_special_tokens=False).input_ids + [EOS_ID]
    ar_seqs.append(ids)
    diff_pairs.append((pt[0], at[0]))

for d_name, is_seen, clauses, c_tensors in dom_data:
    if is_seen:
        ids = [BOS_ID] + tokenizer("".join(clauses), add_special_tokens=False).input_ids + [EOS_ID]
        ar_seqs.append(ids)
    else:
        # For Unseen Domains, models ONLY see individual clauses (or c0->c1 premise), NEVER c1->c2!
        ids_p = [BOS_ID] + tokenizer(clauses[0] + clauses[1], add_special_tokens=False).input_ids + [EOS_ID]
        ids_c2 = [BOS_ID] + tokenizer(clauses[2], add_special_tokens=False).input_ids + [EOS_ID]
        ar_seqs.append(ids_p); ar_seqs.append(ids_c2)
        diff_pairs.append((pad_prompt(clauses[0])[0], c_tensors[1][0][0]))

# Train AR & Pure Diffusion (150 steps)
opt_ar = torch.optim.AdamW(m_ar.parameters(), lr=3e-4)
opt_diff = torch.optim.AdamW(m_diff.parameters(), lr=3e-4)
dp_p = torch.stack([x[0] for x in diff_pairs]); dp_a = torch.stack([x[1] for x in diff_pairs])

for step in range(1, 151):
    opt_ar.zero_grad()
    loss_a = sum(m_ar(torch.tensor([s], device='cuda'), labels=torch.tensor([s], device='cuda')).loss for s in ar_seqs) / len(ar_seqs)
    loss_a.backward(); torch.nn.utils.clip_grad_norm_(m_ar.parameters(), 1.0); opt_ar.step()

    opt_diff.zero_grad()
    loss_d = m_diff.forward_loss(dp_p, dp_a)
    loss_d.backward(); torch.nn.utils.clip_grad_norm_(m_diff.parameters(), 1.0); opt_diff.step()

# Helper to train DualBrain (512 & 256) and PureThought (512)
def train_dual_brain(model_inst, steps=200, train_fast_muscle=True):
    opt_i = torch.optim.AdamW(model_inst.parameters(), lr=3e-4)
    model_inst.train()
    N_dom = len(dom_data)  # 6 domains
    for step in range(1, steps + 1):
        lr = 3e-5 + 0.5 * (3e-4 - 3e-5) * (1 + math.cos(math.pi * step / steps))
        for pg in opt_i.param_groups: pg['lr'] = lr
        opt_i.zero_grad()
        loss_total = 0.0

        if train_fast_muscle:
            s1_p, s1_c, s1_l = [], [], []
            for pt, at, _, _ in rote_data:
                for m in (torch.rand(SLOT_LEN, device='cuda') < 0.5, torch.ones(SLOT_LEN, dtype=torch.bool, device='cuda')):
                    if not m.any(): m[0] = True
                    c = at[0].clone(); c[m] = MASK_ID
                    l = at[0].clone(); l[~m] = -100
                    s1_p.append(pt[0]); s1_c.append(c); s1_l.append(l)
            logits_s1 = model_inst.forward_fast_muscle(torch.stack(s1_p), torch.stack(s1_c))
            loss_total = loss_total + F.cross_entropy(logits_s1.reshape(-1, logits_s1.size(-1)), torch.stack(s1_l).reshape(-1), ignore_index=-100)
        else:
            # Pure Thought Model: forces Rote Facts through the 16-D fuzzy thought bottleneck!
            rt_q_t, rt_q_v, rt_a_t = [], [], []
            for _, at, q_s, _ in rote_data:
                qt, qv = pad_slot(q_s)
                rt_q_t.append(qt[0]); rt_q_v.append(qv[0]); rt_a_t.append(at[0])
            dq_r, sq_r = model_inst.encode_thought(torch.stack(rt_q_t), torch.stack(rt_q_v))
            # Add noisy/masked versions for thought rendering of rote facts
            canv_r = torch.full_like(torch.stack(rt_a_t), MASK_ID)
            logits_r = model_inst.forward_thought_render(dq_r, sq_r, canv_r)
            loss_total = loss_total + F.cross_entropy(logits_r.reshape(-1, logits_r.size(-1)), torch.stack(rt_a_t).reshape(-1), ignore_index=-100)

        # Unsupervised Slow-Feature Domain Invariance + Orthogonal Structure across 6 Domains x 4 States
        all_ct, all_cv = [], []
        for _, _, _, c_tensors in dom_data:
            for ct, cv in c_tensors:
                all_ct.append(ct[0]); all_cv.append(cv[0])
        b_ct = torch.stack(all_ct); b_cv = torch.stack(all_cv)
        d_vec, s_vec = model_inst.encode_thought(b_ct, b_cv)
        d_grid = d_vec.view(N_dom, 4, -1)
        s_grid = s_vec.view(N_dom, 4, -1)

        loss_dis = 0.0
        for dom_i in range(N_dom):
            for st_j in range(1, 4):
                loss_dis = loss_dis + (1.0 - (d_grid[dom_i, 0] * d_grid[dom_i, st_j]).sum())
        for d_a in range(N_dom):
            for d_b in range(d_a + 1, N_dom):
                loss_dis = loss_dis + F.relu((d_grid[d_a, 0] * d_grid[d_b, 0]).sum() + 0.1)

        for st_j in range(4):
            for dom_i in range(1, N_dom):
                loss_dis = loss_dis + (1.0 - (s_grid[0, st_j] * s_grid[dom_i, st_j]).sum())
        for st_a in range(4):
            for st_b in range(st_a + 1, 4):
                for dom_i in range(N_dom):
                    loss_dis = loss_dis + F.relu((s_grid[dom_i, st_a] * s_grid[dom_i, st_b]).sum() + 0.1)

        # Train causal law hopping ONLY on the 2 Seen Domains (Water & Highway)! NEVER on the 4 Unseen Domains!
        s_seen = s_grid[:2, :, :]
        pred_s = model_inst.forward_struct_hop(s_seen[:, :-1, :])
        loss_hop = (1.0 - (pred_s * s_seen[:, 1:, :].detach()).sum(dim=-1)).mean()

        s2_d, s2_s, s2_c, s2_l = [], [], [], []
        for idx in range(N_dom * 4):
            for m in (torch.rand(SLOT_LEN, device='cuda') < 0.5, torch.ones(SLOT_LEN, dtype=torch.bool, device='cuda')):
                if not m.any(): m[0] = True
                c = b_ct[idx].clone(); c[m] = MASK_ID
                l = b_ct[idx].clone(); l[~m] = -100
                s2_d.append(d_vec[idx]); s2_s.append(s_vec[idx]); s2_c.append(c); s2_l.append(l)

        logits_s2 = model_inst.forward_thought_render(torch.stack(s2_d), torch.stack(s2_s), torch.stack(s2_c))
        loss_s2 = F.cross_entropy(logits_s2.reshape(-1, logits_s2.size(-1)), torch.stack(s2_l).reshape(-1), ignore_index=-100)

        loss_total = loss_total + loss_s2 + 2.0 * loss_hop + 0.3 * loss_dis
        loss_total.backward()
        torch.nn.utils.clip_grad_norm_(model_inst.parameters(), 1.0)
        opt_i.step()
    model_inst.eval()

print("Training PureThought-512, DualBrain-512, and Tiny-DualBrain-256...", flush=True)
train_dual_brain(m_pure_thought, steps=180, train_fast_muscle=False)
train_dual_brain(m_dual_512, steps=200, train_fast_muscle=True)
train_dual_brain(m_dual_256, steps=220, train_fast_muscle=True)
m_ar.eval(); m_diff.eval()

# ==============================================================================
# EVALUATE ALL 5 MODELS ON THE 3 BENCHMARK SUITES:
# Suite 1: 6 Exact Rote Memories (Phone numbers, IP, Server Key) -> Exact string match
# Suite 2: 2 Seen Domain Causal Reasoning -> Exact clause match
# Suite 3: 4 UNSEEN Domain Zero-Shot Analogy Reasoning (醫學/網路/工廠/呼吸) -> Exact clause match
# ==============================================================================
def eval_all():
    results = {}
    for name_m, mod in [
        ("1. AR-MiniMind (8L, 512D)", m_ar),
        ("2. Pure-Diffusion (8L, 512D)", m_diff),
        ("3. Pure-Thought LCM (8L, 512D)", m_pure_thought),
        ("4. Dual-Brain (8L, 512D)", m_dual_512),
        ("5. Tiny Dual-Brain (4L, 256D)", m_dual_256)
    ]:
        rote_ok = 0
        seen_ok = 0
        unseen_ok = 0
        unseen_samples = []

        # 1. Test 6 Rote Memories
        for q_s, gt_a in rote_facts:
            if mod is m_ar:
                inp = torch.tensor([[BOS_ID] + tokenizer(q_s, add_special_tokens=False).input_ids], device='cuda')
                out = mod.generate(inp, max_new_tokens=10, do_sample=False, use_cache=True)
                pred_s = clean_str(out[0, inp.size(1):])
            elif mod is m_diff:
                out = mod.generate_slots(pad_prompt(q_s), steps=2)
                pred_s = clean_str(out[0])
            elif mod is m_pure_thought:
                qt, qv = pad_slot(q_s)
                dq, sq = mod.encode_thought(qt, qv)
                canv = torch.full((1, SLOT_LEN), MASK_ID, dtype=torch.long, device='cuda')
                logits = mod.forward_thought_render(dq, sq, canv)
                pred_s = clean_str(logits.argmax(dim=-1)[0])
            else:
                _, out = mod.probe_muscle(pad_prompt(q_s), steps=2)
                pred_s = clean_str(out[0])
            if pred_s == gt_a:
                rote_ok += 1

        # 2. Test 2 Seen Domains + 4 Unseen Domains
        for d_name, is_seen, clauses, c_tensors in dom_data:
            p_str = clauses[0] + clauses[1]
            gt_c2 = clauses[2]
            if mod is m_ar:
                inp = torch.tensor([[BOS_ID] + tokenizer(p_str, add_special_tokens=False).input_ids], device='cuda')
                out = mod.generate(inp, max_new_tokens=10, do_sample=False, use_cache=True)
                pred_s = clean_str(out[0, inp.size(1):])
            elif mod is m_diff:
                out = mod.generate_slots(pad_prompt(p_str), steps=2)
                pred_s = clean_str(out[0])
            elif mod is m_pure_thought:
                out = mod.solve_thought_analogy(c_tensors[0][0], c_tensors[0][1], c_tensors[1][0], c_tensors[1][1], steps=2)
                pred_s = clean_str(out[0])
            else:
                min_c, fast_out = mod.probe_muscle(pad_prompt(p_str), steps=2)
                if min_c >= 0.85:
                    pred_s = clean_str(fast_out[0])
                else:
                    out = mod.solve_thought_analogy(c_tensors[0][0], c_tensors[0][1], c_tensors[1][0], c_tensors[1][1], steps=2)
                    pred_s = clean_str(out[0])

            if is_seen:
                if gt_c2[:6] in pred_s: seen_ok += 1
            else:
                if gt_c2[:6] in pred_s: unseen_ok += 1
                unseen_samples.append((d_name, pred_s, gt_c2))

        total_solved = rote_ok + seen_ok + unseen_ok
        p_m = count_m(mod)
        density = total_solved / p_m  # Solved Tasks per Million Parameters!
        results[name_m] = (p_m, rote_ok, seen_ok, unseen_ok, total_solved, density, unseen_samples)
    return results

res = eval_all()
print("\n====================================================================================================================", flush=True)
print("=== [GRAND BENCHMARK] 5-WAY ARCHITECTURE & KNOWLEDGE DENSITY COMPARISON (Same Data, Tested on Unseen Analogy!) ===", flush=True)
print("====================================================================================================================", flush=True)
print(f"{'Model Architecture':<32} | {'Params':>7} | {'Rote (6)':>8} | {'Seen (2)':>8} | {'Unseen IQ (4)':>13} | {'Total (12)':>10} | {'Knowledge Density':>18}", flush=True)
print("-" * 116, flush=True)
for k, (pm, r_ok, s_ok, u_ok, tot, dens, _) in res.items():
    print(f"{k:<32} | {pm:6.2f}M | {r_ok:5d}/6  | {s_ok:5d}/2  | {u_ok:9d}/4   | {tot:6d}/12  | {dens:11.2f} tasks/M", flush=True)

print("\n=== [Zero-Shot Unseen Domain Predictions: 5.6M Tiny Dual-Brain vs 30.0M AR-MiniMind vs 30.0M Pure-Diffusion] ===", flush=True)
for idx_u in range(4):
    d_name = res["5. Tiny Dual-Brain (4L, 256D)"][6][idx_u][0]
    gt_u   = res["5. Tiny Dual-Brain (4L, 256D)"][6][idx_u][2]
    ar_u   = res["1. AR-MiniMind (8L, 512D)"][6][idx_u][1]
    df_u   = res["2. Pure-Diffusion (8L, 512D)"][6][idx_u][1]
    du_u   = res["5. Tiny Dual-Brain (4L, 256D)"][6][idx_u][1]
    print(f"Unseen Domain [{d_name}] Ground Truth: '{gt_u}'", flush=True)
    print(f"  -> 30.0M AR-MiniMind         : '{ar_u}'", flush=True)
    print(f"  -> 30.0M Pure-Diffusion      : '{df_u}'", flush=True)
    print(f"  ->  5.6M Tiny Dual-Brain (Ours): '{du_u}'", flush=True)

torch.save(m_dual_256.state_dict(), '/content/minimind/out/tiny_dual_brain_256.pth')


Model Sizes: AR-512=30.03M | Diff-512=30.03M | PureThought-512=30.08M | DualBrain-512=31.93M | Tiny-DualBrain-256=5.46M


Training PureThought-512, DualBrain-512, and Tiny-DualBrain-256...


=== [GRAND BENCHMARK] 5-WAY ARCHITECTURE & KNOWLEDGE DENSITY COMPARISON (Same Data, Tested on Unseen Analogy!) ===


Model Architecture               |  Params | Rote (6) | Seen (2) | Unseen IQ (4) | Total (12) |  Knowledge Density


--------------------------------------------------------------------------------------------------------------------


...

  ->  5.6M Tiny Dual-Brain (Ours): '产线积压負荷急剧升高，'


Unseen Domain [呼吸系統] Ground Truth: '呼吸道阻力就会急剧升高，'


  -> 30.0M AR-MiniMind         : ''


  -> 30.0M Pure-Diffusion      : '�胀变�当当气管管当当'


  ->  5.6M Tiny Dual-Brain (Ours): '呼吸道阻力就会急剧升高，'
